# ICD Single-Case RAG Baseline

Use this notebook to score one case at a time against the ICD manual search index. It supports either a free-text manual summary or a row fetched from `usdo_aa_catalog.research_tam_datasets.mimic_icd10_note_dataset_2017_2019_strict`, and logs retrieval plus coding artifacts to MLflow.

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import UTC, datetime
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env", override=False)

from baselines.icd_rag import load_config, score_prediction
from baselines.icd_rag.single_case import (
    build_single_case_chain,
    fetch_case_record,
    format_retrieved_chunks,
    normalize_prediction,
    retrieve_candidate_chunks,
)

config = load_config()
PROMPTS_DIR = REPO_ROOT / "baselines" / "icd_rag" / "prompts"
STRICT_TABLE = "usdo_aa_catalog.research_tam_datasets.mimic_icd10_note_dataset_2017_2019_strict"

try:
    spark  # type: ignore[name-defined]
except NameError:
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()

print(f"execution_env: {config.execution_env}")
print(f"search service: {config.search_service.name}")
print(f"search index: {config.index_name}")
print(f"prompts dir: {PROMPTS_DIR}")
print(f"strict table: {STRICT_TABLE}")

In [ ]:
MODEL_NAME = "openai:gpt-5"
MLFLOW_EXPERIMENT = "/Users/sonutka@mfcgd.com/icd-rag-single-case"
MLFLOW_RUN_NAME = f"icd_rag_single_case_{datetime.now(UTC).strftime('%Y%m%d_%H%M%S')}"
TOP_K = 8
LLM_MAX_ATTEMPTS = 2

TARGET_HADM_ID = 20037934

MANUAL_SUMMARY = None
MANUAL_EXPECTED_CODES = []

print("Set TARGET_HADM_ID in this cell to the admission you want to score.")
print("hadm_id is unique in this strict table, so it is enough by itself.")
print("Leave MANUAL_SUMMARY = None to fetch from the table, or set it for free-text testing.")

In [ ]:
preview_columns = [
    column_name
    for column_name in [
        "hadm_id",
        "subject_id",
        "note_id",
        "real_discharge_year_min",
        "real_discharge_year_max",
        "output_icd_codes",
    ]
    if column_name in spark.table(STRICT_TABLE).columns
]
preview_df = spark.table(STRICT_TABLE).select(*preview_columns).limit(5).toPandas()
preview_df

In [ ]:
if MANUAL_SUMMARY:
    case_record = {
        "hadm_id": None,
        "subject_id": None,
        "note_id": None,
        "case_summary": MANUAL_SUMMARY.strip(),
        "expected_icd_codes": [str(code).strip().upper() for code in MANUAL_EXPECTED_CODES if str(code).strip()],
    }
else:
    if TARGET_HADM_ID is None:
        raise ValueError("Set TARGET_HADM_ID in the config cell before running this cell.")
    case_record = fetch_case_record(
        spark=spark,
        table_name=STRICT_TABLE,
        hadm_id=TARGET_HADM_ID,
    )

case_metadata = {
    "hadm_id": case_record.get("hadm_id"),
    "subject_id": case_record.get("subject_id"),
    "note_id": case_record.get("note_id"),
    "expected_icd_codes": case_record.get("expected_icd_codes", []),
}

pd.DataFrame([case_metadata])

In [ ]:
print(case_record["case_summary"][:4000])

In [ ]:
if mlflow.active_run() is not None:
    mlflow.end_run()

mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name=MLFLOW_RUN_NAME) as run:
    mlflow.langchain.autolog(log_traces=True, silent=True)
    mlflow.log_params(
        {
            "model_name": MODEL_NAME,
            "search_service": config.search_service.name,
            "search_index": config.index_name,
            "strict_table": STRICT_TABLE,
            "top_k": TOP_K,
            "llm_max_attempts": LLM_MAX_ATTEMPTS,
            "case_hadm_id": "manual" if case_record.get("hadm_id") is None else int(case_record["hadm_id"]),
            "case_subject_id": "manual" if case_record.get("subject_id") is None else int(case_record["subject_id"]),
            "case_note_id": "manual" if case_record.get("note_id") is None else str(case_record["note_id"]),
        }
    )
    mlflow.set_tags(
        {
            "baseline": "icd_rag_single_case",
            "prompt_template": "single_case_coding.j2",
            "source_table": STRICT_TABLE,
        }
    )

    with mlflow.start_span(name="single_case_icd_rag") as case_span:
        case_span.set_inputs(
            {
                "hadm_id": case_record.get("hadm_id"),
                "subject_id": case_record.get("subject_id"),
                "note_id": case_record.get("note_id"),
                "model_name": MODEL_NAME,
                "search_service": config.search_service.name,
                "search_index": config.index_name,
                "top_k": TOP_K,
                "case_summary_preview": case_record["case_summary"][:2000],
            }
        )

        with mlflow.start_span(name="retrieve_context") as retrieval_span:
            retrieval_span.set_inputs(
                {
                    "query_text_preview": case_record["case_summary"][:2000],
                    "top_k": TOP_K,
                    "search_service": config.search_service.name,
                    "search_index": config.index_name,
                }
            )
            retrieved_chunks = retrieve_candidate_chunks(
                config=config,
                summary_text=case_record["case_summary"],
                top_k=TOP_K,
            )
            retrieval_span.set_outputs(
                {
                    "retrieved_chunk_count": len(retrieved_chunks),
                    "top_hits": [
                        {
                            "chunk_id": chunk.get("chunk_id"),
                            "source_type": chunk.get("source_type"),
                            "chunk_title": chunk.get("chunk_title"),
                            "code": chunk.get("code"),
                            "score": chunk.get("score"),
                            "text_preview": (chunk.get("text") or "")[:500],
                        }
                        for chunk in retrieved_chunks[:5]
                    ],
                }
            )

        retrieved_context_text = format_retrieved_chunks(retrieved_chunks)

        with mlflow.start_span(name="predict_icd_codes") as predict_span:
            predict_span.set_inputs(
                {
                    "model_name": MODEL_NAME,
                    "prompt_template": "single_case_coding.j2",
                    "retrieved_chunk_count": len(retrieved_chunks),
                    "retrieved_chunk_ids": [chunk.get("chunk_id") for chunk in retrieved_chunks[:10]],
                    "case_summary_preview": case_record["case_summary"][:2000],
                }
            )
            chain = build_single_case_chain(
                model_name=MODEL_NAME,
                prompts_dir=PROMPTS_DIR,
                llm_max_attempts=LLM_MAX_ATTEMPTS,
            )
            raw_prediction = chain.invoke(
                {
                    "case_summary": case_record["case_summary"],
                    "retrieved_context": retrieved_context_text,
                    "top_k": TOP_K,
                }
            )
            prediction = normalize_prediction(raw_prediction)
            predict_span.set_outputs(
                {
                    "predicted_icd_codes": prediction.get("predicted_icd_codes", []),
                    "confidence": prediction.get("confidence"),
                    "rationale": prediction.get("rationale", ""),
                    "supporting_evidence": prediction.get("supporting_evidence", []),
                }
            )

        expected_codes = case_record.get("expected_icd_codes", [])
        metrics = score_prediction(prediction["predicted_icd_codes"], expected_codes) if expected_codes else {}
        case_span.set_outputs(
            {
                "status": "ok",
                "retrieved_chunk_count": len(retrieved_chunks),
                "predicted_icd_codes": prediction.get("predicted_icd_codes", []),
                "metrics": metrics,
            }
        )

    artifact_payload = {
        "case_record": case_record,
        "retrieved_chunks": retrieved_chunks,
        "prediction": prediction,
        "metrics": metrics,
        "run_id": run.info.run_id,
    }
    mlflow.log_dict(artifact_payload, "single_case/run_summary.json")
    mlflow.log_dict({"retrieved_chunks": retrieved_chunks}, "single_case/retrieved_chunks.json")
    mlflow.log_dict(prediction, "single_case/prediction.json")
    for metric_name, metric_value in metrics.items():
        mlflow.log_metric(metric_name, float(metric_value))

    run_id = run.info.run_id

result_bundle = {
    "run_id": run_id,
    "prediction": prediction,
    "metrics": metrics,
    "retrieved_chunks": retrieved_chunks,
}

print(json.dumps({k: v for k, v in result_bundle.items() if k != "retrieved_chunks"}, indent=2))

In [ ]:
prediction_df = pd.DataFrame(
    [
        {
            "predicted_icd_codes": ", ".join(result_bundle["prediction"].get("predicted_icd_codes", [])),
            "confidence": result_bundle["prediction"].get("confidence"),
            "rationale": result_bundle["prediction"].get("rationale", ""),
            "supporting_evidence": " | ".join(result_bundle["prediction"].get("supporting_evidence", [])),
        }
    ]
)
retrieval_df = pd.DataFrame(result_bundle["retrieved_chunks"])
retrieval_df["text_preview"] = retrieval_df["text"].fillna("").str.slice(0, 500)

prediction_df
retrieval_df[["chunk_id", "source_type", "document_title", "chunk_title", "semantic_path", "code", "score", "text_preview"]]